In [1]:
import pandas as pd

In [2]:
sales = pd.read_csv('sales.csv')
breakfast = pd.read_csv('data_breakfast_with_coordinates.csv')
lunch = pd.read_csv('data_lunch_with_coordinates.csv')

/var/folders/z8/9gdx3qqs78j00xkhq1k15pyh0000gn/T/ipykernel_3301/381680460.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  breakfast = pd.read_csv('data_breakfast_with_coordinates.csv')
/var/folders/z8/9gdx3qqs78j00xkhq1k15pyh0000gn/T/ipykernel_3301/381680460.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  lunch = pd.read_csv('data_lunch_with_coordinates.csv')


In [3]:
sales.columns = sales.columns.str.lower()
breakfast.columns = breakfast.columns.str.lower()
lunch.columns = lunch.columns.str.lower()

In [4]:
sales.head()

,time_of_day,school_code,school_name,date,item,description,total,free_meals,reduced_price_meals,full_price_meals,adults,alac_student,alac_adult,earned_student,earned_adult,earned_alac_student,earned_alac_adult,adj_alac,adj_meal
0,breakfast,17,COLVIN_RUN_ELEMENTARY,05/01/2025,1146,CEREAL MEAL,10,2,0,8,0,0,0,0,0,0,0,0,0
1,breakfast,17,COLVIN_RUN_ELEMENTARY,05/01/2025,1170,MINI PANCAKES,32,2,0,30,0,0,0,0,0,0,0,0,0
2,breakfast,17,COLVIN_RUN_ELEMENTARY,05/01/2025,1310,ALC BREAKFAST ENTREE,8,0,0,0,0,8,0,0,0,0,0,0,0
3,breakfast,17,COLVIN_RUN_ELEMENTARY,05/01/2025,1405,EGG & CHEESE ON BISCUIT,2,0,0,2,0,0,0,0,0,0,0,0,0
4,breakfast,17,COLVIN_RUN_ELEMENTARY,05/01/2025,151,CEREAL/ NO MILK,9,0,0,0,0,9,0,0,0,0,0,0,0


In [5]:
breakfast.head()

,school_name,date,identifier,name,planned_reimbursable,planned_non-reimbursable,planned_total,offered_reimbursable,offered_non-reimbursable,offered_total,...,fns area,level,fcps region,cep schools,original_school_name_excel,normalized_school_name_excel,address,zipcode,latitude,longitude
0,Aldrin Elementary,5/01/2025,10041,Mini Maple Pancakes (Package),57,0,57,59,56,0,...,4,ES,Region 1,NaN,Aldrin ES,aldrin es,"11375 Center Harbor Rd, Reston, VA 20194",20194.0,38.979865,-77.337806
1,Aldrin Elementary,5/01/2025,20001,1% White Milk (Each),33,0,33,37,34,0,...,4,ES,Region 1,NaN,Aldrin ES,aldrin es,"11375 Center Harbor Rd, Reston, VA 20194",20194.0,38.979865,-77.337806
2,Aldrin Elementary,5/01/2025,20017,Orange (6 Slices per 1/2 cup),51,0,51,55,53,0,...,4,ES,Region 1,NaN,Aldrin ES,aldrin es,"11375 Center Harbor Rd, Reston, VA 20194",20194.0,38.979865,-77.337806
3,Aldrin Elementary,5/01/2025,20038,Fat Free White Milk (Each),11,0,11,15,13,0,...,4,ES,Region 1,NaN,Aldrin ES,aldrin es,"11375 Center Harbor Rd, Reston, VA 20194",20194.0,38.979865,-77.337806
4,Aldrin Elementary,5/01/2025,30003,Honey Cheerios Cereal (Each),10,0,10,10,7,0,...,4,ES,Region 1,NaN,Aldrin ES,aldrin es,"11375 Center Harbor Rd, Reston, VA 20194",20194.0,38.979865,-77.337806


In [6]:
lunch.head()

,school_name,date,identifier,name,planned_reimbursable,planned_non-reimbursable,planned_total,offered_reimbursable,offered_non-reimbursable,offered_total,...,fns area,level,fcps region,cep schools,original_school_name_excel,normalized_school_name_excel,address,zipcode,latitude,longitude
0,Aldrin Elementary,5/01/2025,10044,Soft Pretzel (1 pretzel),15,0,15,31,27,0,...,4,ES,Region 1,NaN,Aldrin ES,aldrin es,"11375 Center Harbor Rd, Reston, VA 20194",20194.0,38.979865,-77.337806
1,Aldrin Elementary,5/01/2025,10062,Corn (1/4 cup (thawed)),92,0,92,155,152,0,...,4,ES,Region 1,NaN,Aldrin ES,aldrin es,"11375 Center Harbor Rd, Reston, VA 20194",20194.0,38.979865,-77.337806
2,Aldrin Elementary,5/01/2025,20001,1% White Milk (Each),44,0,44,120,43,0,...,4,ES,Region 1,NaN,Aldrin ES,aldrin es,"11375 Center Harbor Rd, Reston, VA 20194",20194.0,38.979865,-77.337806
3,Aldrin Elementary,5/01/2025,20002,Fat Free Chocolate Milk (Each),133,0,133,155,131,0,...,4,ES,Region 1,NaN,Aldrin ES,aldrin es,"11375 Center Harbor Rd, Reston, VA 20194",20194.0,38.979865,-77.337806
4,Aldrin Elementary,5/01/2025,20016,String Cheese (Each),2,0,2,5,2,0,...,4,ES,Region 1,NaN,Aldrin ES,aldrin es,"11375 Center Harbor Rd, Reston, VA 20194",20194.0,38.979865,-77.337806


In [7]:
sales['date'] = pd.to_datetime(sales['date'], errors='coerce')
breakfast['date'] = pd.to_datetime(breakfast['date'], errors='coerce')
lunch['date'] = pd.to_datetime(lunch['date'], errors='coerce')

In [8]:
def clean_numeric(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(r"[\$,%, ]", "", regex=True)
                .replace("nan", "0")
                .astype(float)
            )
    return df

In [9]:
num_cols = [
    "served_non-reimbursable", "discarded_total", "discarded_cost",
    "subtotal_cost", "left_over_percent_of_offered", "left_over_cost",
    "left_over_total", "production_cost_total"
]
breakfast = clean_numeric(breakfast, num_cols)
lunch = clean_numeric(lunch, num_cols)

In [10]:
# --- Find popular items ---
# From sales.csv
popular_sales = (
    sales.groupby(["time_of_day", "description"])["total"]
    .sum()
    .reset_index()
    .sort_values(["time_of_day", "total"], ascending=[True, False])
)

# From breakfast (by served)
popular_breakfast = (
    breakfast.groupby("name")["served_reimbursable"]
    .sum()
    .reset_index()
    .sort_values("served_reimbursable", ascending=False)
)

# From lunch_combined (by served)
popular_lunch = (
    lunch.groupby("name")["served_reimbursable"]
    .sum()
    .reset_index()
    .sort_values("served_reimbursable", ascending=False)
)

# --- Least discarded (lower is better) ---
if "discarded_total" in breakfast.columns:
    popular_breakfast_low_discarded = (
        breakfast.groupby("name")["discarded_total"]
        .sum()
        .reset_index()
        .sort_values("discarded_total", ascending=True)
    )
else:
    popular_breakfast_low_discarded = None

if "discarded_total" in lunch.columns:
    popular_lunch_low_discarded = (
        lunch.groupby("name")["discarded_total"]
        .sum()
        .reset_index()
        .sort_values("discarded_total", ascending=True)
    )
else:
    popular_lunch_low_discarded = None

In [11]:
# --- Results ---
print("Top 10 Breakfast Items (by served_total):")
print(popular_breakfast.head(10))

print("\nTop 10 Lunch Items (by served_total):")
print(popular_lunch.head(10))

if popular_breakfast_low_discarded is not None:
    print("\nTop 10 Breakfast Items (least discarded_total):")
    print(popular_breakfast_low_discarded.head(10))

if popular_lunch_low_discarded is not None:
    print("\nTop 10 Lunch Items (least discarded_total):")
    print(popular_lunch_low_discarded.head(10))

print("\nTop 10 Items (from sales.csv):")
print(popular_sales.groupby("time_of_day").head(10))

# --- Export to CSV files ---
print("\n--- Exporting to CSV files ---")

# Export served_total rankings
popular_breakfast.to_csv('breakfast_popular_by_served.csv', index=False)
popular_lunch.to_csv('lunch_popular_by_served.csv', index=False)
print("Exported: breakfast_popular_by_served.csv, lunch_popular_by_served.csv")

# Export discarded_total rankings (least discarded)
if popular_breakfast_low_discarded is not None:
    popular_breakfast_low_discarded.to_csv('breakfast_popular_by_least_discarded.csv', index=False)
    print("Exported: breakfast_popular_by_least_discarded.csv")
else:
    print("Skipped: breakfast_popular_by_least_discarded.csv (column not found)")

if popular_lunch_low_discarded is not None:
    popular_lunch_low_discarded.to_csv('lunch_popular_by_least_discarded.csv', index=False)
    print("Exported: lunch_popular_by_least_discarded.csv")
else:
    print("Skipped: lunch_popular_by_least_discarded.csv (column not found)")

Top 10 Breakfast Items (by served_total):
                                                  name  served_reimbursable
7                                   Apple Juice (Each)               413120
99                       Orange Tangerine Juice (Each)               223678
83                       Mini Maple Pancakes (Package)               191648
2                                 1% White Milk (Each)               182381
19                                       Bagel (Bagel)                90662
57                                 Cream Cheese (Each)                69351
71                        Honey Cheerios Cereal (Each)                68135
110               Red Delicious Apple (Each - serving)                66184
55   Cinnamon Toast Crunch Cereal, 25% Less Sugar (...                63168
54                         Cinnamon Chex Cereal (Each)                59297

Top 10 Lunch Items (by served_total):
                                     name  served_reimbursable
169        Fat Free 

In [12]:
# The actual amount consumed (served minus what was thrown away)

# For breakfast
breakfast_net_consumption = (
    breakfast.groupby("name")
    .agg({
        'served_reimbursable': 'sum',
        'discarded_total': 'sum'
    })
    .reset_index()
)

# Calculate net consumption
breakfast_net_consumption['net_consumption'] = (
    breakfast_net_consumption['served_reimbursable'] - breakfast_net_consumption['discarded_total']
)

# Sort by net consumption (higher is better)
breakfast_net_consumption = breakfast_net_consumption.sort_values('net_consumption', ascending=False)

# For lunch
lunch_net_consumption = (
    lunch.groupby("name")
    .agg({
        'served_reimbursable': 'sum',
        'discarded_total': 'sum'
    })
    .reset_index()
)

# Calculate net consumption
lunch_net_consumption['net_consumption'] = (
    lunch_net_consumption['served_reimbursable'] - lunch_net_consumption['discarded_total']
)

# Sort by net consumption (higher is better)
lunch_net_consumption = lunch_net_consumption.sort_values('net_consumption', ascending=False)


In [13]:
# --- Display Net Consumption Results ---
print("=== NET CONSUMPTION POPULARITY RANKINGS ===")

print("\nTop 15 Breakfast Items (by Net Consumption):")
print("=" * 90)
breakfast_display = breakfast_net_consumption[['name', 'net_consumption', 'served_reimbursable', 'discarded_total']].head(15)
breakfast_display.columns = ['Item Name', 'Net Consumption', 'Total Served', 'Total Discarded']
print(breakfast_display)

print("\nTop 15 Lunch Items (by Net Consumption):")
print("=" * 90)
lunch_display = lunch_net_consumption[['name', 'net_consumption', 'served_reimbursable', 'discarded_total']].head(15)
lunch_display.columns = ['Item Name', 'Net Consumption', 'Total Served', 'Total Discarded']
print(lunch_display)


=== NET CONSUMPTION POPULARITY RANKINGS ===

Top 15 Breakfast Items (by Net Consumption):
                                             Item Name  Net Consumption  \
7                                   Apple Juice (Each)        413020.16   
99                       Orange Tangerine Juice (Each)        223569.49   
83                       Mini Maple Pancakes (Package)        190833.89   
2                                 1% White Milk (Each)        181925.39   
19                                       Bagel (Bagel)         89294.02   
57                                 Cream Cheese (Each)         69295.26   
71                        Honey Cheerios Cereal (Each)         67916.89   
110               Red Delicious Apple (Each - serving)         65913.46   
55   Cinnamon Toast Crunch Cereal, 25% Less Sugar (...         62809.09   
54                         Cinnamon Chex Cereal (Each)         58894.37   
29                        Blueberry Chex Cereal (Each)         49441.68   
64        

In [14]:
# --- Export Net Consumption Results ---
print("=== EXPORTING NET CONSUMPTION RESULTS ===")

# Export the net consumption rankings
breakfast_net_consumption.to_csv('breakfast_net_consumption_popularity.csv', index=False)
lunch_net_consumption.to_csv('lunch_net_consumption_popularity.csv', index=False)

print("Exported: breakfast_net_consumption_popularity.csv")
print("Exported: lunch_net_consumption_popularity.csv")


=== EXPORTING NET CONSUMPTION RESULTS ===
Exported: breakfast_net_consumption_popularity.csv
Exported: lunch_net_consumption_popularity.csv


In [15]:
# Calculate leftover rate for breakfast items
breakfast_leftover_rate = (
    breakfast.groupby("name")
    .agg({
        'left_over_total': 'sum',
        'offered_reimbursable': 'sum'
    })
    .reset_index()
)

# Calculate leftover rate (percentage)
breakfast_leftover_rate['leftover_rate'] = (
    breakfast_leftover_rate['left_over_total'] / 
    breakfast_leftover_rate['offered_reimbursable'].replace(0, 1)  # Avoid division by zero
) * 100

# Sort by leftover rate (higher = more waste)
breakfast_leftover_rate = breakfast_leftover_rate.sort_values('leftover_rate', ascending=False)

# Calculate leftover rate for lunch items
lunch_leftover_rate = (
    lunch.groupby("name")
    .agg({
        'left_over_total': 'sum',
        'offered_reimbursable': 'sum'
    })
    .reset_index()
)

# Calculate leftover rate (percentage)
lunch_leftover_rate['leftover_rate'] = (
    lunch_leftover_rate['left_over_total'] / 
    lunch_leftover_rate['offered_reimbursable'].replace(0, 1)  # Avoid division by zero
) * 100

# Sort by leftover rate (higher = more waste)
lunch_leftover_rate = lunch_leftover_rate.sort_values('leftover_rate', ascending=False)


In [16]:
# --- Display Leftover Rate Results ---
print("=== LEFTOVER RATE RANKINGS ===")

print("\nTop 15 Breakfast Items (Highest Leftover Rate - Most Waste):")
print("=" * 100)
breakfast_display = breakfast_leftover_rate[['name', 'leftover_rate', 'left_over_total', 'offered_reimbursable']].head(15)
breakfast_display.columns = ['Item Name', 'Leftover Rate (%)', 'Total Left Over', 'Total Offered']
breakfast_display['Leftover Rate (%)'] = breakfast_display['Leftover Rate (%)'].round(2)
print(breakfast_display)

print("\nTop 15 Lunch Items (Highest Leftover Rate - Most Waste):")
print("=" * 100)
lunch_display = lunch_leftover_rate[['name', 'leftover_rate', 'left_over_total', 'offered_reimbursable']].head(15)
lunch_display.columns = ['Item Name', 'Leftover Rate (%)', 'Total Left Over', 'Total Offered']
lunch_display['Leftover Rate (%)'] = lunch_display['Leftover Rate (%)'].round(2)
print(lunch_display)


=== LEFTOVER RATE RANKINGS ===

Top 15 Breakfast Items (Highest Leftover Rate - Most Waste):
                                             Item Name  Leftover Rate (%)  \
62                      Fat Free Chocolate Milk (Each)              98.83   
37   Cereal, Cinnamon Toasters, Malt-O-Meal, IW (Su...              60.26   
64                          Fat Free White Milk (Each)              53.53   
43            Cherry Flavored Dried Cranberries (Each)              53.12   
63        Fat Free Unflavored Shelf-Stable Milk (Each)              52.70   
80   Milk, Lactose Free,  Fat Free, Shelf-Stable (E...              51.24   
96            Orange Flavored Dried Cranberries (Each)              50.00   
132   Watermelon Flavored Dried Cranberries (2 Eaches)              48.05   
17                        Assorted Cereal (Cereal Cup)              47.13   
101             Patty, Chicken, Breaded, WG (2 strips)              46.88   
66                       Frozen, Blueberries (1/2 cup)      

In [17]:
# --- Export Leftover Rate Results ---
print("=== EXPORTING LEFTOVER RATE RESULTS ===")

# Export the full leftover rate analysis
breakfast_leftover_rate.to_csv('breakfast_leftover_rate_analysis.csv', index=False)
lunch_leftover_rate.to_csv('lunch_leftover_rate_analysis.csv', index=False)

print("Exported: breakfast_leftover_rate_analysis.csv")
print("Exported: lunch_leftover_rate_analysis.csv")

# Calculate overall statistics
print("\n=== OVERALL LEFTOVER STATISTICS ===")
breakfast_total_leftover = breakfast['left_over_total'].sum()
breakfast_total_offered = breakfast['offered_reimbursable'].sum()
breakfast_overall_rate = (breakfast_total_leftover / breakfast_total_offered) * 100

lunch_total_leftover = lunch['left_over_total'].sum()
lunch_total_offered = lunch['offered_reimbursable'].sum()
lunch_overall_rate = (lunch_total_leftover / lunch_total_offered) * 100

print(f"Breakfast Overall Leftover Rate: {breakfast_overall_rate:.2f}%")
print(f"  Total Left Over: {breakfast_total_leftover:,.0f}")
print(f"  Total Offered: {breakfast_total_offered:,.0f}")

print(f"\nLunch Overall Leftover Rate: {lunch_overall_rate:.2f}%")
print(f"  Total Left Over: {lunch_total_leftover:,.0f}")
print(f"  Total Offered: {lunch_total_offered:,.0f}")

print(f"\nCombined Overall Leftover Rate: {((breakfast_total_leftover + lunch_total_leftover) / (breakfast_total_offered + lunch_total_offered)) * 100:.2f}%")


=== EXPORTING LEFTOVER RATE RESULTS ===
Exported: breakfast_leftover_rate_analysis.csv
Exported: lunch_leftover_rate_analysis.csv

=== OVERALL LEFTOVER STATISTICS ===
Breakfast Overall Leftover Rate: 20.26%
  Total Left Over: 496,886
  Total Offered: 2,451,946

Lunch Overall Leftover Rate: 15.78%
  Total Left Over: 1,845,563
  Total Offered: 11,695,760

Combined Overall Leftover Rate: 16.56%
